# 🚀 Student Life Data Explorer

**Lumexa Python Foundations · Course 3: Python Tools & Data Foundations · Lesson 8**

| | |
|---|---|
| **Project** | Student Life Data Explorer (portfolio project 9 of 9) |
| **You will practise** | planning, functions, Pandas (cleaning, `groupby`), Matplotlib charts, `assert` tests, README and project documentation |
| **Time** | about 90 minutes |

This is your **Python Foundations capstone**: an end-to-end data project. You will plan it, load a student-life survey, **clean** the messy data, **summarise** it, draw **charts**, answer your own questions, **test** your code and write the documentation that lets someone else understand it.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.

> 🧪 **About the data:** the survey below is **synthetic practice data**. It was invented for this project, so no real student is in it, and it was made messy on purpose so that you can practise cleaning. You can later replace it with your own **anonymous** class survey (ask your teacher first, and use numbers instead of names).

## Part 0 · Plan before you code

### The problem
A school wants to understand how students' daily habits (study, sleep and screen time) relate to their exam scores.

### Audience
Teachers and students who want a clear, honest summary.

### Inputs and outputs
| Input | Process | Output |
|---|---|---|
| A survey CSV with one row per student | Clean it, summarise it, chart it | A cleaned table, summary numbers, three charts and written findings |

### Three questions this project will answer
1. *Do students who sleep more (7 hours or more) score higher in the exam?*
2. *How are study time and exam score related?*
3. *Do the grades (8, 9 and 10) differ in their average score?*

### Requirements
| Priority | Requirement |
|---|---|
| **Must** | Load the survey and report how many rows it has |
| **Must** | Find and fix missing values, duplicates, wrong types and untidy text, and report every change |
| **Must** | Calculate at least five summaries, including two with `groupby` |
| **Must** | Draw at least three charts with titles, labels and a legend where it helps |
| **Must** | Answer the three questions in writing, and state one limit of the data |
| **Should** | Check the cleaning with `assert` tests |
| **Should** | Save the charts as image files |
| **Could** | Let the reader choose a grade and see only that grade |

✏️ **Your turn:** add **two requirements of your own** to the table (double-click to edit).

## ⚙️ Setup

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (7, 4)

Path("docs").mkdir(exist_ok=True)          # the charts will be saved here
print("Ready. Pandas", pd.__version__)

## Part 1 · Load the data and write a data report

The survey is stored as text (this is what a CSV file looks like inside). `io.StringIO` lets Pandas read it like a file. In a real project you would put the file in a `data/` folder and use `pd.read_csv("data/survey.csv")`.

| Column | Meaning |
|---|---|
| `student_id` | An anonymous code (never use real names) |
| `grade` | School grade: 8, 9 or 10 |
| `study_hours` | Hours of study per day |
| `sleep_hours` | Hours of sleep per night |
| `screen_hours` | Hours of screen time per day |
| `club` | Whether the student is in a club or team (Yes / No) |
| `exam_score` | Score in the exam, from 0 to 100 |

In [ ]:
SURVEY_CSV = """student_id,grade,study_hours,sleep_hours,screen_hours,club,exam_score
S01,8,4.0,6.5,3.0,Yes,78
S02,9,3.5,8.0,4.5,yes,74
S03,8,3.5,7.5,4.5,No,63
S04,10,3.0,6.5,3.0,No,63
S05,8,,7.0,3.5,Yes,54
S06,10,4.0,8.0,3.5,No,89
S07,9,2.0,7.5,1.0,Yes,73
S08,9,2.0,5.0,5.0,No,54
S09,9,2.5,7.5,6.0, no,70
S10,8,2.5,,4.0,No,60
S11,10,1.5,7.5,1.5,No,72
S12,9,2.5,7.5,3.5,Yes,61
S13,8,0.5,7.5,n/a,No,54
S14,8,2.0,6.5,2.5,Yes,60
S15,8,2.0,9.0,5.0,No,60
S16,8,3.5,6.5,1.5,NO,75
S17,8,2.5,7.5,3.5,No,72
S18,10,,7.0,4.0,No,76
S19,10,3.0,9.5,4.0,No,74
S20,8,1.5,6.0,5.5,No,53
S21,10,2.5,7.0,2.5,No,69
S22,10,4.0,9.0,,Yes,78
S23,8,3.5,7.5,4.5,Yes,56
S24,10,2.5,6.0,4.0,Yes,50
S25,9,1.0,6.0,4.5,No,46
S26,8,2.0,6.5,4.0,Yes,53
S27,10,2.0,6.5,4.0,Yes ,59
S28,9,2.0,7.5,5.0,Yes,64
S29,9,3.0,8.0,3.5,Yes,70
S30,9,,6.5,4.5,Yes,62
S31,9,4.0,6.5,4.0,No,73
S32,10,1.5,7.0,2.5,No,65
S33,9,4.0,6.5,4.0,No,71
S34,8,3.0,,3.5,No,69
S35,9,1.5,6.5,5.5,Yes,53
S36,8,3.0,7.0,3.0,No,70
S37,8,3.0,6.0,2.5,Yes,105
S38,8,3.5,7.5,4.0,No,64
S39,9,5.5,6.5,8.0,No,69
S40,10,3.5,8.0,3.0,Yes,71
S07,9,2.0,7.5,1.0,Yes,73
S20,8,1.5,6.0,5.5,No,53
"""

def load_data(text):
    """Read the survey data into a DataFrame."""
    return pd.read_csv(io.StringIO(text))

raw = load_data(SURVEY_CSV)
print("Rows and columns:", raw.shape)
raw.head(8)

### 🔍 The data report: what is wrong with it?

Look first, change second. These four checks find the problems:

In [ ]:
print("Missing values in each column:")
print(raw.isna().sum())
print("\nDuplicate rows:", raw.duplicated().sum())
print("\nData types:")
print(raw.dtypes)
print("\nDifferent spellings in the club column:", sorted(raw["club"].unique()))
print("Highest exam score:", raw["exam_score"].max(), "(scores should be 0 to 100!)")

✏️ **Your turn:** list every problem you found in a Markdown cell. There are at least **six**: missing values, a text value in a number column, duplicates, untidy club answers and an impossible score. *(Double-click to write.)*

## Part 2 · Clean the data (one function, with a log)

`clean_data()` returns a **clean copy** (the original `raw` is never changed) and prints a **log** of every decision. Notice that:

- **text** is tidied with `.str.strip()` and `.str.title()`;
- bad numbers become `NaN` with `pd.to_numeric(..., errors="coerce")`;
- **duplicates** are dropped;
- an **impossible score** is treated as missing, and a student without a score is dropped (we cannot guess an exam result);
- missing study, sleep and screen values are **filled with the median**, which is a fair typical value. This is a guess, so we report how many we filled.

In [ ]:
NUMBER_COLUMNS = ["study_hours", "sleep_hours", "screen_hours", "exam_score"]

def clean_data(df):
    """Return a clean copy of the survey and print a log of every change."""
    clean = df.copy()

    # 1. Tidy the club column: Yes / No
    clean["club"] = clean["club"].str.strip().str.title()
    print("• club: removed extra spaces and made the capital letters consistent")

    # 2. Make the number columns real numbers; text such as "n/a" becomes NaN
    for column in NUMBER_COLUMNS:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")
    print("• number columns: converted to numbers (text values became missing)")

    # 3. Remove duplicate rows
    before = len(clean)
    clean = clean.drop_duplicates().copy()
    print(f"• duplicates: removed {before - len(clean)} repeated row(s)")

    # 4. Impossible exam scores are treated as missing, then those rows are dropped
    impossible = (clean["exam_score"] > 100) | (clean["exam_score"] < 0)
    clean.loc[impossible, "exam_score"] = float("nan")
    before = len(clean)
    clean = clean.dropna(subset=["exam_score"]).copy()
    print(f"• exam_score: dropped {before - len(clean)} row(s) with a missing or impossible score")

    # 5. Fill the remaining gaps with the median of each column
    for column in ["study_hours", "sleep_hours", "screen_hours"]:
        gaps = int(clean[column].isna().sum())
        clean[column] = clean[column].fillna(clean[column].median())
        print(f"• {column}: filled {gaps} missing value(s) with the median")

    clean["exam_score"] = clean["exam_score"].astype(int)
    return clean.reset_index(drop=True)

clean = clean_data(raw)
print("\nBefore:", raw.shape, "| After:", clean.shape)
clean.head(8)

In [ ]:
# Before and after, side by side
comparison = pd.DataFrame({
    "before": [len(raw), int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "after": [len(clean), int(clean.isna().sum().sum()), int(clean.duplicated().sum())],
}, index=["rows", "missing values", "duplicate rows"])
comparison

## Part 3 · Test the cleaning

A test is a claim that must be true. If any `assert` fails, the notebook stops with a clear message, so a mistake cannot hide.

In [ ]:
assert clean.isna().sum().sum() == 0, "there are still missing values"
assert not clean.duplicated().any(), "there are still duplicate rows"
assert clean["exam_score"].between(0, 100).all(), "an exam score is outside 0-100"
assert set(clean["club"].unique()) == {"Yes", "No"}, "the club column still has untidy answers"
assert len(clean) == 39, "unexpected number of rows after cleaning"
print("✅ All 5 tests passed.")

## Part 4 · Summaries

Summary numbers turn 39 rows into a few sentences. `groupby` splits the rows into groups and summarises each one. Read it aloud: *"group by grade, take the exam scores and find the mean."*

In [ ]:
def summarise(df):
    """Return the main results of the survey as a dictionary."""
    well_rested = df["sleep_hours"] >= 7
    return {
        "students": len(df),
        "average_score": round(df["exam_score"].mean(), 1),
        "median_score": df["exam_score"].median(),
        "highest_score": int(df["exam_score"].max()),
        "score_by_grade": df.groupby("grade")["exam_score"].mean().round(1),
        "score_by_club": df.groupby("club")["exam_score"].mean().round(1),
        "score_well_rested": round(df[well_rested]["exam_score"].mean(), 1),
        "score_less_rested": round(df[~well_rested]["exam_score"].mean(), 1),
        "students_well_rested": int(well_rested.sum()),
    }

results = summarise(clean)
print("Students:", results["students"])
print("Average exam score:", results["average_score"], "| median:", results["median_score"], "| highest:", results["highest_score"])
print("\nAverage score by grade:")
print(results["score_by_grade"])
print("\nAverage score by club membership:")
print(results["score_by_club"])
print(f"\nSleeping 7+ hours ({results['students_well_rested']} students): {results['score_well_rested']}")
print(f"Sleeping less than 7 hours: {results['score_less_rested']}")

## Part 5 · Charts

A chart without a **title**, **axis labels** and (when there is more than one series) a **legend** is only a drawing. We make three charts, one for each kind you learned: **bar**, **scatter** and **line**. Each one is saved in the `docs/` folder so you can use it in your README.

### Chart 1 · Bar chart: average score by grade

In [ ]:
by_grade = results["score_by_grade"]
plt.bar(by_grade.index.astype(str), by_grade.values, color="#4C78A8", label="Average exam score")
plt.title("Average exam score by grade")
plt.xlabel("Grade")
plt.ylabel("Exam score (0-100)")
plt.ylim(0, 100)
plt.legend()
plt.savefig("docs/score_by_grade.png", bbox_inches="tight")
plt.show()

### Chart 2 · Scatter plot: study time and exam score

Each dot is one student. We colour the students by club membership, so the **legend** is needed.

In [ ]:
for club, colour in [("Yes", "#F58518"), ("No", "#4C78A8")]:
    group = clean[clean["club"] == club]
    plt.scatter(group["study_hours"], group["exam_score"], color=colour, label=f"Club: {club}", alpha=0.8)
plt.title("Study time and exam score")
plt.xlabel("Study time (hours per day)")
plt.ylabel("Exam score (0-100)")
plt.legend()
plt.savefig("docs/study_vs_score.png", bbox_inches="tight")
plt.show()

### Chart 3 · Line chart: score and hours of sleep

We round the sleep hours to whole hours and average the score for each. A **line** is right here because the hours of sleep are in a natural order.

In [ ]:
clean["sleep_rounded"] = clean["sleep_hours"].round().astype(int)
by_sleep = clean.groupby("sleep_rounded")["exam_score"].mean()

plt.plot(by_sleep.index, by_sleep.values, marker="o", color="#54A24B", label="Average exam score")
plt.title("Average exam score by hours of sleep")
plt.xlabel("Sleep (hours per night, rounded)")
plt.ylabel("Average exam score (0-100)")
plt.legend()
plt.savefig("docs/score_by_sleep.png", bbox_inches="tight")
plt.show()
print("Saved charts:", sorted(p.name for p in Path("docs").glob("*.png")))

## Part 6 · Answer the questions

The sentences below are built from your **results**, so they always match the data. Read them, then write your own conclusions in the next Markdown cell.

In [ ]:
difference = results["score_well_rested"] - results["score_less_rested"]
print(f"Q1 · Students who sleep 7+ hours scored {results['score_well_rested']} on average, against {results['score_less_rested']} for the others, a difference of {difference:+.1f} points.")

correlation = clean["study_hours"].corr(clean["exam_score"])     # between -1 and +1
if correlation > 0.3:
    pattern = "more study hours go with higher scores"
elif correlation < -0.3:
    pattern = "more study hours go with lower scores"
else:
    pattern = "there is no clear pattern between study hours and scores"
print(f"Q2 · In this sample the correlation between study time and score is {correlation:.2f}: {pattern}.")

best_grade = by_grade.idxmax()
print(f"Q3 · Grade {best_grade} has the highest average score ({by_grade.max()}); the lowest is grade {by_grade.idxmin()} ({by_grade.min()}).")

print()
print("⚠️  Limits of this data: only", results["students"], "students; some values were filled in with the median; the data is synthetic.")
print("⚠️  A relationship is not a proof: sleeping more does not necessarily CAUSE a higher score.")

### ✍️ My findings

*(Double-click to write at least five sentences. Say what the three charts show, answer the three questions in your own words, and explain one limit of the data.)*

-
-
-

## ✏️ Your turn · Go one step further

Add **two analyses of your own**. Ideas: Do club members score differently from non-members *within* each grade? What is the average screen time for each grade? Which five students have the lowest sleep, and what are their scores? Make **one extra chart** (with a title, labels and a legend) and save it in `docs/`.

In [ ]:
# ✏️ Your turn: analysis 1

In [ ]:
# ✏️ Your turn: analysis 2

## Part 7 · Document and share

A project is finished when **someone else can understand and run it**. Copy the template below into a file called `README.md` in your project folder, and fill it in.

````markdown
# Student Life Data Explorer

A data project that explores how sleep, study time and screen time relate to exam scores.

## What it shows
A short description of your three questions, with one chart image from `docs/`.

## How to run it
1. Open `student_life_data_explorer.ipynb` in Google Colab or Jupyter.
2. Choose **Runtime → Restart session and run all**.

## The data
Where it came from (synthetic practice data), how many rows (42 raw, 39 after cleaning) and what each column means.

## What I did to clean it
List every change: duplicates, missing values, text tidied, impossible scores.

## What I found
Three findings, and one limit of the data.

## What I learned
Two things you learned and one thing you would do next.
````

### ✅ Final checklist

- [ ] The notebook runs from the top with **Restart session and run all**.
- [ ] All five `assert` tests pass.
- [ ] Every chart has a title, axis labels and (where it helps) a legend.
- [ ] The findings answer all three questions and mention one limit.
- [ ] There are **no names, passwords or keys** anywhere in the project.
- [ ] My Git history has at least **five commits** with clear messages (for example: *Add data loading*, *Clean missing values*, *Add score chart*, *Write findings*, *Write README*).
- [ ] If I am 13 or older and a parent or guardian agrees, I have a plan for publishing the project on GitHub. Otherwise I will share the notebook link with my teacher.

## 🏆 Stretch challenges

1. Move `load_data`, `clean_data` and `summarise` into a file `analysis.py` and import them (Lesson 2).
2. Add a function that takes a **grade** and shows only that grade's summary and charts.
3. Collect your own **anonymous** survey from classmates (with your teacher's permission), save it as `data/survey.csv` and run your project on it.

## 🪞 Reflection

- What was the hardest part of the project, and how did you solve it?
- What decision about the data would you explain differently if a teacher asked you about it?
- What would you do next with more time?

*Your reflection:*

## 📤 How to submit

Choose **File → Save a copy in Drive**, then **Share → Anyone with the link can view**, and send the link to your teacher together with your **two-minute demo plan**: the problem, what you built, what you found and what you would do next.